In [2]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from transform.tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests



d:\eosc_dc\metadata-warehouse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [4]:
user = "admin"
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [18]:
query_runs_sql = """
SELECT
id
FROM harvest_runs
WHERE status = 'closed'
"""

In [23]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_runs_sql)
        
        results = cur.fetchall()
        print(results)

        ids = [str(res['id']) for res in results]
        print(ids)


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': UUID('0ae7e781-7a68-45cc-ad61-4bd4819dca30')}, {'id': UUID('fd045493-52f9-4bcb-8265-9d7628ac48fd')}, {'id': UUID('a0f3e9d6-3955-4e98-ad8e-267b30c9da41')}, {'id': UUID('ff8a126a-3365-450e-a39d-55cba03790a8')}]
['0ae7e781-7a68-45cc-ad61-4bd4819dca30', 'fd045493-52f9-4bcb-8265-9d7628ac48fd', 'a0f3e9d6-3955-4e98-ad8e-267b30c9da41', 'ff8a126a-3365-450e-a39d-55cba03790a8']


In [26]:
BASE_URL = "http://localhost:8080/index"
index_name = "test_datacite"

In [28]:
for harvest_run_id in ids:
    params = {
        "harvest_run_id": harvest_run_id,
        "index_name": index_name
    }
    response = requests.get(BASE_URL, params=params)
    print(harvest_run_id, response.status_code, response.text)

0ae7e781-7a68-45cc-ad61-4bd4819dca30 200 {"number_of_batches":2}
fd045493-52f9-4bcb-8265-9d7628ac48fd 200 {"number_of_batches":397}
a0f3e9d6-3955-4e98-ad8e-267b30c9da41 200 {"number_of_batches":7}
ff8a126a-3365-450e-a39d-55cba03790a8 200 {"number_of_batches":39}
